In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 7
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L2hvbWUvbWF4L2F1bGEtcXVhbnR1bS9jb250ZW50LzA0LXF1Yml0'
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/importlib/_bootstrap.py": 1787687117.0310943, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/importlib/_bootstrap_external.py": 1787687117.0310943, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/codecs.py": 1787687116.7980936, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/encodings/aliases.py": 1787687116.852094, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/encodings/__init__.py": 1787687116.8510938, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/encodings/utf_8.py": 1787687116.888094, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/abc.py": 1787687116.7790935, "/home/max/.local/share/uv/python/cpython-3.10-linux-x86_64-gnu/lib/python3.10/io.py": 1787687117.0350945, "/home/max/.local/share/uv/python/cpython-3.10-linu

In [2]:
# Importando as bibliotecas necessárias
# Não se preocupe se não entender todas agora - vamos usá-las ao longo do notebook

import numpy as np                      # Para operações com matrizes e vetores
import sympy as sp                      # Para matemática simbólica (fórmulas bonitas)
from sympy import Matrix, symbols       # Para criar matrizes e símbolos matemáticos
from IPython.display import Markdown    # Para exibir fórmulas formatadas
from qiskit.quantum_info import Statevector  # Para trabalhar com estados quânticos
from matplotlib import pyplot as plt    # Para criar gráficos
from qiskit.visualization import plot_bloch_multivector, plot_state_qsphere  # Visualizações quânticas

In [3]:
# Vamos criar esses estados básicos usando SymPy (biblioteca de matemática simbólica)
# Isso nos permite trabalhar com fórmulas matemáticas de forma elegante

# Estado |0⟩ - primeiro número é 1, segundo é 0
ket_0 = Matrix([[1], 
                [0]])

# Estado |1⟩ - primeiro número é 0, segundo é 1
ket_1 = Matrix([[0], 
                [1]])

# Exibir de forma bonita com LaTeX (notação matemática profissional)
display(Markdown(f"$$\\ket{{0}} = {sp.latex(ket_0)}$$"))
display(Markdown(f"$$\\ket{{1}} = {sp.latex(ket_1)}$$"))

print("\n✅ Estados básicos criados com sucesso!")

$$\ket{0} = \left[\begin{matrix}1\\0\end{matrix}\right]$$

$$\ket{1} = \left[\begin{matrix}0\\1\end{matrix}\right]$$


✅ Estados básicos criados com sucesso!


In [4]:
# Agora vamos criar os MESMOS estados usando NumPy (só para fins de comparação de bibliotecas)
# NumPy é mais eficiente para cálculos numéricos (com números reais)

# Estado |0⟩ usando NumPy
q_zero = np.array([
    [1],   # Probabilidade 100% de medir 0
    [0]    # Probabilidade 0% de medir 1
])

# Estado |1⟩ usando NumPy
q_one = np.array([
    [0],   # Probabilidade 0% de medir 0
    [1]    # Probabilidade 100% de medir 1
])

print("=== Estado |0⟩ ===")
print(q_zero)
print("\n=== Estado |1⟩ ===")
print(q_one)

print("\n💡 Dica: Esses vetores representam a 'probabilidade' de medir cada resultado!")

=== Estado |0⟩ ===
[[1]
 [0]]

=== Estado |1⟩ ===
[[0]
 [1]]

💡 Dica: Esses vetores representam a 'probabilidade' de medir cada resultado!


In [5]:
# Vamos criar um estado genérico em superposição!
# Usamos símbolos 𝛼 (alpha) e 𝛽 (beta) para representar valores desconhecidos

alpha, beta = symbols('alpha beta', complex=True)   # Definindo 𝛼 e 𝛽 como números complexos
psi = alpha * ket_0 + beta * ket_1                  # Estado genérico em superposição

print("Estado quântico genérico criado:")

display(Markdown(f"$$\\ket{{\\psi}} = {sp.latex(psi)}$$"))

print("\n💡 Isso significa:")
print("   - Com probabilidade |α|², medimos 0")
print("   - Com probabilidade |β|², medimos 1")
print("   - Antes da medição, o qubit está em AMBOS os estados!")

Estado quântico genérico criado:


$$\ket{\psi} = \left[\begin{matrix}\alpha\\\beta\end{matrix}\right]$$


💡 Isso significa:
   - Com probabilidade |α|², medimos 0
   - Com probabilidade |β|², medimos 1
   - Antes da medição, o qubit está em AMBOS os estados!


In [6]:
# Vamos testar a porta X na prática com Python!

# Definir a matriz da porta X
X = Matrix([[0, 1],
            [1, 0]])

print("🔄 Aplicando a Porta X...\n")

# Aplicar X ao estado |0⟩
result_0 = X * ket_0        # ket_0 é o estado |0⟩ com o vetor definido no código lá atrás, lembra?
print("Teste 1: X|0⟩ deve dar |1⟩")
display(Markdown(f"$$X \\ket{{0}} = {sp.latex(result_0)}$$ ✅"))

# Aplicar X ao estado |1⟩
result_1 = X * ket_1
print("\nTeste 2: X|1⟩ deve dar |0⟩")
display(Markdown(f"$$X \\ket{{1}} = {sp.latex(result_1)}$$ ✅"))

print("\n💡 A porta X realmente inverte os estados, como esperado!")

🔄 Aplicando a Porta X...

Teste 1: X|0⟩ deve dar |1⟩


$$X \ket{0} = \left[\begin{matrix}0\\1\end{matrix}\right]$$ ✅


Teste 2: X|1⟩ deve dar |0⟩


$$X \ket{1} = \left[\begin{matrix}1\\0\end{matrix}\right]$$ ✅


💡 A porta X realmente inverte os estados, como esperado!


In [7]:
# Agora vamos definir a porta X usando NumPy (outra maneira de fazer a mesma coisa)

gate_x = np.array([
    [0, 1],  # Primeira linha da matriz
    [1, 0]   # Segunda linha da matriz
])

print("Porta X definida com NumPy:")
print(gate_x)
print("\n✅ Pronta para usar em cálculos numéricos!")

Porta X definida com NumPy:
[[0 1]
 [1 0]]

✅ Pronta para usar em cálculos numéricos!


In [8]:
# Vamos aplicar a porta X ao estado |0⟩ e ver o resultado numérico

print("=== Aplicando Porta X ao estado |0⟩ ===\n")

# Multiplicação de matriz por vetor: np.dot(matriz, vetor)
novo_estado = np.dot(gate_x, q_zero)

print("Estado inicial |0⟩:")
print(q_zero)

print("\nApós aplicar a porta X:")
print(novo_estado)

print("\n✅ Esperávamos |1⟩ (vetor [0, 1]), e foi isso que obtivemos!")
print("💡 A porta X funcionou como um 'interruptor' que inverte o qubit!")

=== Aplicando Porta X ao estado |0⟩ ===

Estado inicial |0⟩:
[[1]
 [0]]

Após aplicar a porta X:
[[0]
 [1]]

✅ Esperávamos |1⟩ (vetor [0, 1]), e foi isso que obtivemos!
💡 A porta X funcionou como um 'interruptor' que inverte o qubit!


In [9]:
# Exemplo de aplicação da porta H (Hadamard)

# Definir a porta H
H = (1 / sp.sqrt(2)) * Matrix([[1, 1],
                               [1, -1]])

# Aplicar a porta H ao estado |0⟩
result_h0 = H * ket_0

# Aplicar a porta H ao estado |1⟩
result_h1 = H * ket_1

# Explicar o resultado utilizando Markdown para formatação bonita (perfumaria)

print("=== Aplicando Porta H (Hadamard) ===\n")
print("A porta H cria superposição dos estados |0⟩ e |1⟩.\n")
display(Markdown(f"$$H \\ket{{0}} = {sp.latex(H)} \\cdot {sp.latex(ket_0)} = {sp.latex(result_h0)}$$"))

print("\nResultado detalhado:")
display(Markdown(f"$${sp.latex(result_h0)} = \\frac{{1}}{{\\sqrt{{2}}}} \\ket{{0}} + \\frac{{1}}{{\\sqrt{{2}}}} \\ket{{1}}$$"))

print("\nAgora aplicando H ao estado |1⟩:\n")
display(Markdown(f"$$H \\ket{{1}} = {sp.latex(H)} \\cdot {sp.latex(ket_1)} = {sp.latex(result_h1)}$$"))

print("\nResultado detalhado:")
display(Markdown(f"$${sp.latex(result_h1)} = \\frac{{1}}{{\\sqrt{{2}}}} \\ket{{0}} - \\frac{{1}}{{\\sqrt{{2}}}} \\ket{{1}}$$"))

=== Aplicando Porta H (Hadamard) ===

A porta H cria superposição dos estados |0⟩ e |1⟩.



$$H \ket{0} = \left[\begin{matrix}\frac{\sqrt{2}}{2} & \frac{\sqrt{2}}{2}\\\frac{\sqrt{2}}{2} & - \frac{\sqrt{2}}{2}\end{matrix}\right] \cdot \left[\begin{matrix}1\\0\end{matrix}\right] = \left[\begin{matrix}\frac{\sqrt{2}}{2}\\\frac{\sqrt{2}}{2}\end{matrix}\right]$$


Resultado detalhado:


$$\left[\begin{matrix}\frac{\sqrt{2}}{2}\\\frac{\sqrt{2}}{2}\end{matrix}\right] = \frac{1}{\sqrt{2}} \ket{0} + \frac{1}{\sqrt{2}} \ket{1}$$


Agora aplicando H ao estado |1⟩:



$$H \ket{1} = \left[\begin{matrix}\frac{\sqrt{2}}{2} & \frac{\sqrt{2}}{2}\\\frac{\sqrt{2}}{2} & - \frac{\sqrt{2}}{2}\end{matrix}\right] \cdot \left[\begin{matrix}0\\1\end{matrix}\right] = \left[\begin{matrix}\frac{\sqrt{2}}{2}\\- \frac{\sqrt{2}}{2}\end{matrix}\right]$$


Resultado detalhado:


$$\left[\begin{matrix}\frac{\sqrt{2}}{2}\\- \frac{\sqrt{2}}{2}\end{matrix}\right] = \frac{1}{\sqrt{2}} \ket{0} - \frac{1}{\sqrt{2}} \ket{1}$$

In [10]:
# Outra maneira de definir a porta H usando NumPy (faz a mesma coisa que o código anterior)
# Porta H (Hadamard): Cria superposição
# Matriz: (1/√2) * [[1, 1], [1, -1]]
gate_h = (1 / np.sqrt(2)) * np.array([
    [1, 1],
    [1, -1]
])

# APLICANDO A PORTA HADAMARD em |0>
# Operação: H * |0> = Superposição
estado_superposicao = np.dot(gate_h, q_zero)

print("\n--- Após aplicar a Porta Hadamard (H) ---")
print(f"Estado de Superposição:\n{estado_superposicao}")
# Note que os valores são aprox 0.707 (que é 1/raiz(2))


--- Após aplicar a Porta Hadamard (H) ---
Estado de Superposição:
[[0.70710678]
 [0.70710678]]


In [11]:
# Exemplo de produto interno

bra_0 = ket_0.T
bra_1 = ket_1.T

# ⟨0|0⟩
phi_00 = bra_0 * ket_0

# ⟨0|1⟩
phi_01 = bra_0 * ket_1

# ⟨1|0⟩
phi_10 = bra_1 * ket_0

# ⟨1|1⟩
phi_11 = bra_1 * ket_1

display(Markdown(f"$$\\langle 0 | 0 \\rangle = {sp.latex(bra_0)} * {sp.latex(ket_0)} = {phi_00[0,0]}$$"))
display(Markdown(f"$$\\langle 0 | 1 \\rangle = {sp.latex(bra_0)} * {sp.latex(ket_1)} = {phi_01[0,0]}$$"))
display(Markdown(f"$$\\langle 1 | 0 \\rangle = {sp.latex(bra_1)} * {sp.latex(ket_0)} = {phi_10[0,0]}$$"))
display(Markdown(f"$$\\langle 1 | 1 \\rangle = {sp.latex(bra_1)} * {sp.latex(ket_1)} = {phi_11[0,0]}$$"))

$$\langle 0 | 0 \rangle = \left[\begin{matrix}1 & 0\end{matrix}\right] * \left[\begin{matrix}1\\0\end{matrix}\right] = 1$$

$$\langle 0 | 1 \rangle = \left[\begin{matrix}1 & 0\end{matrix}\right] * \left[\begin{matrix}0\\1\end{matrix}\right] = 0$$

$$\langle 1 | 0 \rangle = \left[\begin{matrix}0 & 1\end{matrix}\right] * \left[\begin{matrix}1\\0\end{matrix}\right] = 0$$

$$\langle 1 | 1 \rangle = \left[\begin{matrix}0 & 1\end{matrix}\right] * \left[\begin{matrix}0\\1\end{matrix}\right] = 1$$

In [12]:
# Exemplo de produto tensorial

# |0⟩ ⊗ |0⟩
tensor_00 = sp.kronecker_product(ket_0, ket_0)
display(
    Markdown(
        f"$$\\ket{{0}} \\otimes \\ket{{0}} = {sp.latex(ket_0)} \\otimes {sp.latex(ket_0)} = \\begin{{bmatrix}} 1 \\cdot {sp.latex(ket_0)} \\\\ 0 \\cdot {sp.latex(ket_0)} \\end{{bmatrix}} = {sp.latex(tensor_00)} = \\ket{{00}}$$"
    )
)

# |0⟩ ⊗ |1⟩
tensor_01 = sp.kronecker_product(ket_0, ket_1)
display(
    Markdown(
        f"$$\\ket{{0}} \\otimes \\ket{{1}} = {sp.latex(ket_0)} \\otimes {sp.latex(ket_1)} = \\begin{{bmatrix}} 1 \\cdot {sp.latex(ket_1)} \\\\ 0 \\cdot {sp.latex(ket_1)} \\end{{bmatrix}} = {sp.latex(tensor_01)} = \\ket{{01}}$$"
    )
)

# |1⟩ ⊗ |0⟩
tensor_10 = sp.kronecker_product(ket_1, ket_0)
display(
    Markdown(
        f"$$\\ket{{1}} \\otimes \\ket{{0}} = {sp.latex(ket_1)} \\otimes {sp.latex(ket_0)} = \\begin{{bmatrix}} 0 \\cdot {sp.latex(ket_0)} \\\\ 1 \\cdot {sp.latex(ket_0)} \\end{{bmatrix}} = {sp.latex(tensor_10)} = \\ket{{10}}$$"
    )
)

# |1⟩ ⊗ |1⟩
tensor_11 = sp.kronecker_product(ket_1, ket_1)
display(
    Markdown(
        f"$$\\ket{{1}} \\otimes \\ket{{1}} = {sp.latex(ket_1)} \\otimes {sp.latex(ket_1)} = \\begin{{bmatrix}} 0 \\cdot {sp.latex(ket_1)} \\\\ 1 \\cdot {sp.latex(ket_1)} \\end{{bmatrix}} = {sp.latex(tensor_11)} = \\ket{{11}}$$"
    )
)

$$\ket{0} \otimes \ket{0} = \left[\begin{matrix}1\\0\end{matrix}\right] \otimes \left[\begin{matrix}1\\0\end{matrix}\right] = \begin{bmatrix} 1 \cdot \left[\begin{matrix}1\\0\end{matrix}\right] \\ 0 \cdot \left[\begin{matrix}1\\0\end{matrix}\right] \end{bmatrix} = \left[\begin{matrix}1\\0\\0\\0\end{matrix}\right] = \ket{00}$$

$$\ket{0} \otimes \ket{1} = \left[\begin{matrix}1\\0\end{matrix}\right] \otimes \left[\begin{matrix}0\\1\end{matrix}\right] = \begin{bmatrix} 1 \cdot \left[\begin{matrix}0\\1\end{matrix}\right] \\ 0 \cdot \left[\begin{matrix}0\\1\end{matrix}\right] \end{bmatrix} = \left[\begin{matrix}0\\1\\0\\0\end{matrix}\right] = \ket{01}$$

$$\ket{1} \otimes \ket{0} = \left[\begin{matrix}0\\1\end{matrix}\right] \otimes \left[\begin{matrix}1\\0\end{matrix}\right] = \begin{bmatrix} 0 \cdot \left[\begin{matrix}1\\0\end{matrix}\right] \\ 1 \cdot \left[\begin{matrix}1\\0\end{matrix}\right] \end{bmatrix} = \left[\begin{matrix}0\\0\\1\\0\end{matrix}\right] = \ket{10}$$

$$\ket{1} \otimes \ket{1} = \left[\begin{matrix}0\\1\end{matrix}\right] \otimes \left[\begin{matrix}0\\1\end{matrix}\right] = \begin{bmatrix} 0 \cdot \left[\begin{matrix}0\\1\end{matrix}\right] \\ 1 \cdot \left[\begin{matrix}0\\1\end{matrix}\right] \end{bmatrix} = \left[\begin{matrix}0\\0\\0\\1\end{matrix}\right] = \ket{11}$$

In [13]:
# Implementação do uso da porta CNOT

# Definindo a porta CNOT
CNOT = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
])  

# |11> 

q_11 = np.array([
    [0],
    [0],
    [0],
    [1]
])

# Aplicando a porta CNOT em |11>
novo_estado_cnot = np.dot(CNOT, q_11)
print("\n--- Após aplicar a Porta CNOT em |11> ---")
print(f"Esperamos |10>, obtivemos:\n{novo_estado_cnot}")

# Aplicando a porta CNOT em |10>
q_10 = np.array([
    [0],
    [0],
    [1],
    [0]
])

novo_estado_cnot_10 = np.dot(CNOT, q_10)
print("\n--- Após aplicar a Porta CNOT em |10> ---")
print(f"Esperamos |11>, obtivemos:\n{novo_estado_cnot_10}")


--- Após aplicar a Porta CNOT em |11> ---
Esperamos |10>, obtivemos:
[[0]
 [0]
 [1]
 [0]]

--- Após aplicar a Porta CNOT em |10> ---
Esperamos |11>, obtivemos:
[[0]
 [0]
 [0]
 [1]]


In [14]:
# Implementação da porta CZ (Controlled-Z)

CZ = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, -1]
])

# |11>
q_11 = np.array([
    [0],
    [0],
    [0],
    [1]
])

# Aplicando a porta CZ em |11>
novo_estado_cz = np.dot(CZ, q_11)
print("\n--- Após aplicar a Porta CZ em |11> ---")
print(f"Esperamos -|11>, obtivemos:\n{novo_estado_cz}")

# Vamos testar em |10>
q_10 = np.array([
    [0],
    [0],
    [1],
    [0]
])  

novo_estado_cz_10 = np.dot(CZ, q_10)
print("\n--- Após aplicar a Porta CZ em |10> ---")
print(f"Esperamos |10>, obtivemos:\n{novo_estado_cz_10}")
print("\n✅ A porta CZ funcionou como esperado! (a porta só inverte o sinal de |11>)")


--- Após aplicar a Porta CZ em |11> ---
Esperamos -|11>, obtivemos:
[[ 0]
 [ 0]
 [ 0]
 [-1]]

--- Após aplicar a Porta CZ em |10> ---
Esperamos |10>, obtivemos:
[[0]
 [0]
 [1]
 [0]]

✅ A porta CZ funcionou como esperado! (a porta só inverte o sinal de |11>)


In [15]:
# Implementação da Porta ZZ

ZZ = np.array([
    [1, 0, 0, 0],
    [0, -1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, 1]
])

# Testando a porta ZZ em |01>
q_01 = np.array([
    [0],
    [1],
    [0],
    [0]
])

novo_estado_zz = np.dot(ZZ, q_01)
print("\n--- Após aplicar a Porta ZZ em |01> ---")
print(f"Esperamos -|01>, obtivemos:\n{novo_estado_zz}")
print("\n✅ A porta ZZ funcionou como esperado! (inverte o sinal de |01> e |10>)")


--- Após aplicar a Porta ZZ em |01> ---
Esperamos -|01>, obtivemos:
[[ 0]
 [-1]
 [ 0]
 [ 0]]

✅ A porta ZZ funcionou como esperado! (inverte o sinal de |01> e |10>)


In [16]:
# A Medição (O Colapso)
# Na teoria: A probabilidade de medir 0 ou 1 é o quadrado da amplitude (módulo ao quadrado).

def medir_qubit(estado_vetor):
    # Extrair amplitudes (alpha e beta)
    alpha = estado_vetor[0][0]
    beta = estado_vetor[1][0]
    
    # Calcular probabilidades (Born Rule): |amplitude|^2
    prob_0 = abs(alpha) ** 2
    prob_1 = abs(beta) ** 2
    
    print(f"\nProbabilidade calculada de ser 0: {prob_0:.2f}")
    print(f"Probabilidade calculada de ser 1: {prob_1:.2f}")
    
    # Simular o "lance de dados" da natureza
    resultado = np.random.choice([0, 1], p=[prob_0, prob_1])
    return resultado

# Vamos medir nosso estado de superposição
leitura = medir_qubit(estado_superposicao)
print(f"Resultado da Medição (Colapso): |{leitura}>")


Probabilidade calculada de ser 0: 0.50
Probabilidade calculada de ser 1: 0.50
Resultado da Medição (Colapso): |1>
